# 🗺️ A* Pathfinding on Real-World Roads

An instrumented, from-scratch implementation of **A\*** that runs on a *real*
road network pulled live from OpenStreetMap — not a toy grid. This notebook:

- Downloads an actual street graph for a place you choose
- Implements A\* (and Dijkstra, as a special case) **from scratch** with `heapq`
- Records the full search process so we can visualize *how* A\* explores
- Compares weighted A\* variants (speed vs. optimality trade-off)
- Generates turn-by-turn directions from the raw graph
- Renders a polished, interactive, dark-themed map of the final route

---

## How A* Works

A\* explores a graph outward from the start node, always expanding the node
that looks most promising next. "Most promising" is measured by:

$$f(n) = g(n) + w \cdot h(n)$$

- **g(n)** — the actual distance travelled from the start to node *n*
- **h(n)** — a heuristic *estimate* of the remaining distance from *n* to the goal
  (here: straight-line / haversine distance — it never overestimates on a
  road network, which guarantees the optimal path when *w = 1*)
- **w** — a weight on the heuristic. `w = 0` degrades A\* into plain
  **Dijkstra** (explores uniformly in all directions). `w = 1` is classic,
  optimal A\*. `w > 1` is **weighted A\*** — faster, but may sacrifice
  optimality by trusting the heuristic more aggressively.

We'll implement this once, parameterized by `w`, and reuse it for every
experiment below.

## 1 · Setup

In [1]:
# Run once if these aren't already installed:
# %pip install osmnx networkx folium geopy pandas matplotlib

In [ ]:
import heapq
import math
import time
from dataclasses import dataclass, field

import folium
import networkx as nx
import osmnx as ox
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from matplotlib.colors import Normalize

ox.settings.log_console = False
ox.settings.use_cache = True

%matplotlib inline
plt.style.use("dark_background")

ModuleNotFoundError: No module named 'folium'

## 2 · Configuration

Edit these to point at any city and any two real locations inside it.
Addresses are geocoded automatically — landmark names, hotel names, and
college names generally work fine.

In [ ]:
PLACE_NAME   = "Pune, Maharashtra, India"
START_ADDR   = "Pimpri Chinchwad College of Engineering, Pune"
END_ADDR     = "Shaniwar Wada, Pune"
NETWORK_TYPE = "drive"        # "drive", "walk", or "bike"
OUTPUT_HTML  = "astar_route_map.html"

## 3 · Download the Real Road Network

`osmnx` pulls the actual OpenStreetMap street graph for the chosen place.
Every node is a real intersection (with lat/lon); every edge is a real road
segment (with a `length` in meters and often a street `name`).

In [ ]:
print(f"Downloading road network for: {PLACE_NAME} ...")
t0 = time.perf_counter()

G = ox.graph_from_place(PLACE_NAME, network_type=NETWORK_TYPE)
if not nx.get_edge_attributes(G, "length"):
    G = ox.add_edge_lengths(G)

print(f"Done in {time.perf_counter() - t0:.1f}s")
print(f"Nodes (intersections): {G.number_of_nodes():,}")
print(f"Edges (road segments): {G.number_of_edges():,}")

## 4 · Geocode Start & End, Snap to Nearest Graph Nodes

In [ ]:
start_ll = ox.geocode(START_ADDR)   # (lat, lon)
end_ll   = ox.geocode(END_ADDR)

start_node = ox.nearest_nodes(G, start_ll[1], start_ll[0])
end_node   = ox.nearest_nodes(G, end_ll[1], end_ll[0])

print(f"Start : {START_ADDR}  ->  {start_ll}  (node {start_node})")
print(f"End   : {END_ADDR}  ->  {end_ll}  (node {end_node})")

## 5 · The Heuristic: Haversine Distance

Straight-line distance over the Earth's curved surface — admissible for
road networks since no road is ever shorter than a straight line between
its endpoints.

In [ ]:
def haversine(lat1, lon1, lat2, lon2):
    """Great-circle distance between two lat/lon points, in meters."""
    R = 6371000
    phi1, phi2 = math.radians(lat1), math.radians(lat2)
    d_phi = math.radians(lat2 - lat1)
    d_lambda = math.radians(lon2 - lon1)
    a = (math.sin(d_phi / 2) ** 2
         + math.cos(phi1) * math.cos(phi2) * math.sin(d_lambda / 2) ** 2)
    return 2 * R * math.asin(math.sqrt(a))

# sanity check: straight-line distance between our two points
print(f"{haversine(*start_ll, *end_ll)/1000:.2f} km as the crow flies")

## 6 · A* Implementation (Instrumented)

Same algorithm as a plain A\*, but it also records the **order** in which
nodes are expanded and the frontier size at each step — purely so we can
visualize the search process later. `weight` lets this same function act
as Dijkstra (`weight=0`), classic A\* (`weight=1`), or weighted/greedy A\*
(`weight>1`).

In [ ]:
@dataclass
class SearchResult:
    path: list
    length_m: float
    elapsed_s: float
    expansion_order: list = field(default_factory=list)   # nodes, in the order visited
    frontier_sizes: list = field(default_factory=list)     # len(open set) at each pop


def astar(graph, start, goal, weight=1.0):
    def h(n):
        y1, x1 = graph.nodes[n]["y"], graph.nodes[n]["x"]
        y2, x2 = graph.nodes[goal]["y"], graph.nodes[goal]["x"]
        return haversine(y1, x1, y2, x2)

    t0 = time.perf_counter()
    open_heap = [(weight * h(start), start)]
    g_score = {start: 0.0}
    came_from = {}
    visited = set()
    expansion_order = []
    frontier_sizes = []

    while open_heap:
        frontier_sizes.append(len(open_heap))
        _, current = heapq.heappop(open_heap)

        if current in visited:
            continue
        visited.add(current)
        expansion_order.append(current)

        if current == goal:
            path = [current]
            while current in came_from:
                current = came_from[current]
                path.append(current)
            path.reverse()
            return SearchResult(path, g_score[goal], time.perf_counter() - t0,
                                 expansion_order, frontier_sizes)

        for neighbor in graph.neighbors(current):
            edge_data = graph.get_edge_data(current, neighbor)
            edge_len = min(e.get("length", 1.0) for e in edge_data.values())
            tentative_g = g_score[current] + edge_len

            if tentative_g < g_score.get(neighbor, math.inf):
                came_from[neighbor] = current
                g_score[neighbor] = tentative_g
                f_score = tentative_g + weight * h(neighbor)
                heapq.heappush(open_heap, (f_score, neighbor))

    return SearchResult(None, math.inf, time.perf_counter() - t0,
                         expansion_order, frontier_sizes)

## 7 · Run A* and Dijkstra, Side by Side

`weight=0` turns off the heuristic entirely — that's exactly Dijkstra's
algorithm, exploring uniformly in every direction instead of being pulled
toward the goal.

In [ ]:
result_astar    = astar(G, start_node, end_node, weight=1.0)
result_dijkstra = astar(G, start_node, end_node, weight=0.0)

print("A*        :", f"{result_astar.length_m/1000:.2f} km, "
      f"{len(result_astar.expansion_order):,} nodes expanded, "
      f"{result_astar.elapsed_s*1000:.1f} ms")
print("Dijkstra  :", f"{result_dijkstra.length_m/1000:.2f} km, "
      f"{len(result_dijkstra.expansion_order):,} nodes expanded, "
      f"{result_dijkstra.elapsed_s*1000:.1f} ms")

## 8 · Benchmark Table

In [ ]:
summary = pd.DataFrame([
    {"Algorithm": "Dijkstra (w=0)", "Distance (km)": result_dijkstra.length_m/1000,
     "Nodes expanded": len(result_dijkstra.expansion_order),
     "Time (ms)": result_dijkstra.elapsed_s*1000},
    {"Algorithm": "A* (w=1)", "Distance (km)": result_astar.length_m/1000,
     "Nodes expanded": len(result_astar.expansion_order),
     "Time (ms)": result_astar.elapsed_s*1000},
]).round(2)

summary

## 9 · Weighted A*: The Speed vs. Optimality Trade-off

Cranking `w` above 1 makes the search greedier — it expands far fewer
nodes (faster) but the heuristic's estimate can mislead it into a
slightly longer path. This sweep makes that trade-off visible.

In [ ]:
weights = [0.0, 0.5, 1.0, 1.5, 2.0, 3.0, 5.0]
sweep_rows = []

for w in weights:
    r = astar(G, start_node, end_node, weight=w)
    sweep_rows.append({
        "weight": w,
        "distance_km": r.length_m / 1000,
        "nodes_expanded": len(r.expansion_order),
        "time_ms": r.elapsed_s * 1000,
    })

sweep_df = pd.DataFrame(sweep_rows)
sweep_df

In [ ]:
fig, ax1 = plt.subplots(figsize=(9, 5))

ax1.plot(sweep_df["weight"], sweep_df["nodes_expanded"], "o-",
         color="#00e5ff", label="Nodes expanded")
ax1.set_xlabel("Heuristic weight (w)")
ax1.set_ylabel("Nodes expanded", color="#00e5ff")
ax1.tick_params(axis="y", labelcolor="#00e5ff")

ax2 = ax1.twinx()
ax2.plot(sweep_df["weight"], sweep_df["distance_km"], "s--",
         color="#ff6b6b", label="Route distance (km)")
ax2.set_ylabel("Route distance (km)", color="#ff6b6b")
ax2.tick_params(axis="y", labelcolor="#ff6b6b")

plt.title("Weighted A*: fewer nodes explored vs. route optimality")
fig.tight_layout()
plt.show()

## 10 · Visualizing the Search Wavefront

Every node A\* expanded, colored by *when* it was expanded (dark = early,
bright = late). Notice how it fans out toward the goal rather than
exploring uniformly like Dijkstra would.

In [ ]:
def plot_wavefront(graph, result, start_ll, end_ll, title):
    fig, ax = ox.plot_graph(graph, show=False, close=False, node_size=0,
                             edge_color="#2b2f3a", edge_linewidth=0.5,
                             bgcolor="#0e0e12", figsize=(10, 10))

    xs = [graph.nodes[n]["x"] for n in result.expansion_order]
    ys = [graph.nodes[n]["y"] for n in result.expansion_order]
    order = range(len(result.expansion_order))

    sc = ax.scatter(xs, ys, c=order, cmap="plasma", s=6, alpha=0.8, zorder=3)

    if result.path:
        path_xs = [graph.nodes[n]["x"] for n in result.path]
        path_ys = [graph.nodes[n]["y"] for n in result.path]
        ax.plot(path_xs, path_ys, color="#00ffcc", linewidth=3, zorder=4,
                label="Final route")

    ax.scatter([start_ll[1]], [start_ll[0]], c="lime", s=120, marker="*",
               zorder=5, label="Start")
    ax.scatter([end_ll[1]], [end_ll[0]], c="red", s=120, marker="*",
               zorder=5, label="End")

    plt.colorbar(sc, ax=ax, label="Expansion order (early -> late)", shrink=0.7)
    ax.legend(loc="upper left")
    ax.set_title(title, color="white")
    plt.show()

plot_wavefront(G, result_astar, start_ll, end_ll,
               f"A* wavefront — {len(result_astar.expansion_order):,} nodes expanded")
plot_wavefront(G, result_dijkstra, start_ll, end_ll,
               f"Dijkstra wavefront — {len(result_dijkstra.expansion_order):,} nodes expanded")

## 11 · Turn-by-Turn Directions

Derived straight from the graph: bearing between consecutive nodes tells
us when the route turns, and each edge's `name` attribute (when OSM has
it) gives us the street name.

In [ ]:
def bearing(lat1, lon1, lat2, lon2):
    phi1, phi2 = math.radians(lat1), math.radians(lat2)
    d_lambda = math.radians(lon2 - lon1)
    x = math.sin(d_lambda) * math.cos(phi2)
    y = (math.cos(phi1) * math.sin(phi2)
         - math.sin(phi1) * math.cos(phi2) * math.cos(d_lambda))
    return (math.degrees(math.atan2(x, y)) + 360) % 360


def edge_name(graph, u, v):
    data = graph.get_edge_data(u, v)
    name = list(data.values())[0].get("name", "unnamed road")
    if isinstance(name, list):
        name = name[0]
    return name


def turn_instructions(graph, path):
    instructions = []
    prev_bearing = None
    seg_start_idx = 0

    for i in range(len(path) - 1):
        u, v = path[i], path[i + 1]
        y1, x1 = graph.nodes[u]["y"], graph.nodes[u]["x"]
        y2, x2 = graph.nodes[v]["y"], graph.nodes[v]["x"]
        b = bearing(y1, x1, y2, x2)

        if prev_bearing is not None:
            delta = (b - prev_bearing + 540) % 360 - 180
            if abs(delta) > 25:
                seg_nodes = path[seg_start_idx:i + 1]
                dist = sum(
                    min(e.get("length", 1.0) for e in graph.get_edge_data(seg_nodes[j], seg_nodes[j+1]).values())
                    for j in range(len(seg_nodes) - 1)
                )
                turn = "left" if delta < 0 else "right"
                if abs(delta) > 100:
                    turn = "sharp " + turn
                instructions.append(
                    f"Continue {dist:.0f} m, then turn {turn} onto "
                    f"{edge_name(graph, path[i], path[i+1])}"
                )
                seg_start_idx = i

        prev_bearing = b

    remaining = path[seg_start_idx:]
    if len(remaining) > 1:
        dist = sum(
            min(e.get("length", 1.0) for e in graph.get_edge_data(remaining[j], remaining[j+1]).values())
            for j in range(len(remaining) - 1)
        )
        instructions.append(f"Continue {dist:.0f} m to arrive at your destination")

    return instructions


for i, step in enumerate(turn_instructions(G, result_astar.path), 1):
    print(f"{i:>2}. {step}")

## 12 · The Final, Polished Interactive Map

Dark base map, glowing double-stroke route, 1 km distance markers,
start/end flags, and a toggle between street and satellite imagery.

In [ ]:
def build_map(graph, result, start_ll, end_ll):
    route_coords = [(graph.nodes[n]["y"], graph.nodes[n]["x"]) for n in result.path]
    mid = route_coords[len(route_coords) // 2]

    m = folium.Map(location=mid, zoom_start=14, tiles=None, control_scale=True)
    folium.TileLayer("CartoDB dark_matter", name="Dark").add_to(m)
    folium.TileLayer("OpenStreetMap", name="Streets").add_to(m)
    folium.TileLayer(
        tiles="https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}",
        attr="Esri", name="Satellite",
    ).add_to(m)

    # glow effect: wide translucent line underneath a thin bright line
    folium.PolyLine(route_coords, color="#00e5ff", weight=14, opacity=0.25).add_to(m)
    folium.PolyLine(route_coords, color="#00e5ff", weight=5, opacity=0.95,
                     tooltip=f"A* route — {result.length_m/1000:.2f} km").add_to(m)

    # 1 km distance markers along the route
    cum_dist, next_marker_km = 0.0, 1
    for i in range(len(route_coords) - 1):
        y1, x1 = route_coords[i]
        y2, x2 = route_coords[i + 1]
        seg = haversine(y1, x1, y2, x2)
        cum_dist += seg
        if cum_dist >= next_marker_km * 1000:
            folium.CircleMarker(
                route_coords[i + 1], radius=5, color="#ffffff",
                fill=True, fill_color="#00e5ff", fill_opacity=1,
                tooltip=f"{next_marker_km} km",
            ).add_to(m)
            next_marker_km += 1

    folium.Marker(start_ll, tooltip="Start",
                  icon=folium.Icon(color="green", icon="play", prefix="fa")).add_to(m)
    folium.Marker(end_ll, tooltip="Destination",
                  icon=folium.Icon(color="red", icon="flag-checkered", prefix="fa")).add_to(m)

    info_html = f'''
    <div style="position: fixed; bottom: 20px; left: 20px; z-index: 9999;
                background: rgba(20,20,25,0.9); color: #eee; padding: 14px 18px;
                border-radius: 10px; font-family: sans-serif; font-size: 13px;
                box-shadow: 0 4px 12px rgba(0,0,0,0.5);">
        <b style="color:#00e5ff;">A* Route Summary</b><br>
        Distance: {result.length_m/1000:.2f} km<br>
        Nodes expanded: {len(result.expansion_order):,}<br>
        Search time: {result.elapsed_s*1000:.1f} ms
    </div>
    '''
    m.get_root().html.add_child(folium.Element(info_html))
    folium.LayerControl().add_to(m)

    m.save(OUTPUT_HTML)
    return m

route_map = build_map(G, result_astar, start_ll, end_ll)
route_map

## 13 · Summary

- Implemented A\* from scratch, with Dijkstra as the `weight=0` special case
- Ran both on a real OpenStreetMap road network and compared nodes expanded / time / distance
- Swept the heuristic weight to visualize the speed-vs-optimality trade-off
- Rendered the search wavefront and generated turn-by-turn directions from raw graph data
- Produced a polished, layered, interactive map saved to `astar_route_map.html`

**Ideas to extend this further:**
- Add live traffic weighting by scaling `length` with real-time speed data
- Try bidirectional A* (search from both ends, meet in the middle) for a further speedup
- Swap in a different heuristic (e.g. Manhattan on projected coordinates) and compare